# FMG × ReMasker: stronger Google Colab validation

This notebook runs the external ReMasker comparison with the paper's fixed protocol, but with a configurable longer training schedule. It uses Adult Income, COMPAS, and Taiwan Credit; ten fixed seeds; 20% test-only MCAR masking; and RF/XGB/LightGBM/MLP classifiers.

Use a GPU runtime when possible. The default is `EPOCHS=100`; change it to `10` for a smoke test. Results and checkpoints are stored in Google Drive so the run can resume after a runtime reset.


In [1]:
!pip -q install lightgbm xgboost openpyxl timm
!rm -rf /content/remasker
!git clone -q https://github.com/alps-lab/remasker.git /content/remasker


In [2]:
from google.colab import files, drive
from pathlib import Path
import os, json, shutil

EPOCHS = 50
os.environ['EPOCHS'] = str(EPOCHS)
drive.mount('/content/drive')
RESULT_DIR = Path('/content/drive/MyDrive/FMG_colab_results') / f'epochs_{EPOCHS}'
RESULT_DIR.mkdir(exist_ok=True)
os.environ['FMG_OUT_DIR'] = str(RESULT_DIR)
DATA_DIR = Path('/content/data'); DATA_DIR.mkdir(exist_ok=True)
PERSISTENT_DATA = RESULT_DIR / 'input_data'; PERSISTENT_DATA.mkdir(exist_ok=True)
required = ['adult.zip', 'compas-scores-two-years.csv', 'taiwan_credit.csv']
if all((PERSISTENT_DATA / name).exists() for name in required):
    for name in required: shutil.copy2(PERSISTENT_DATA / name, DATA_DIR / name)
    print('Restored input datasets from Drive.')
else:
    uploaded = files.upload()
    for name, blob in uploaded.items(): (DATA_DIR / name).write_bytes(blob)
    def canonicalize(prefix, suffix, target):
        candidates = sorted(DATA_DIR.glob(prefix + '*' + suffix))
        assert candidates, f'Upload a file beginning with {prefix} and ending with {suffix}'
        src = candidates[-1]
        dst = DATA_DIR / target
        if src.resolve() != dst.resolve():
            shutil.copy2(src, dst)
    canonicalize('adult', '.zip', 'adult.zip')
    canonicalize('compas-scores-two-years', '.csv', 'compas-scores-two-years.csv')
    canonicalize('taiwan_credit', '.csv', 'taiwan_credit.csv')
    for name in required: shutil.copy2(DATA_DIR / name, PERSISTENT_DATA / name)
print('Canonical inputs:', sorted(p.name for p in DATA_DIR.iterdir()))
print('Persistent results directory:', RESULT_DIR)

Mounted at /content/drive
Restored input datasets from Drive.
Canonical inputs: ['adult.zip', 'compas-scores-two-years.csv', 'taiwan_credit.csv']
Persistent results directory: /content/drive/MyDrive/FMG_colab_results/epochs_50


In [3]:
# Recreate the exact local helper and runner in Colab.
from pathlib import Path
import json
Path('/content/work/FMG-research/analysis').mkdir(parents=True, exist_ok=True)
Path('/content/work/external/remasker').mkdir(parents=True, exist_ok=True)
Path('/content/work/FMG-research/analysis/_primary_core.py').write_text('"""Shared, notebook-faithful helpers for new FMG analyses.\n\nThis module intentionally mirrors the preprocessing, split, model and test-only\nmasking protocol in FMG_research_final_REVIEWER_FIXED.ipynb. It does not alter\nthe archived results or the source notebook.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import LabelEncoder, MinMaxScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.neural_network import MLPClassifier\nfrom xgboost import XGBClassifier\nfrom lightgbm import LGBMClassifier\n\nSEEDS = (42, 123, 456, 1, 7, 13, 21, 37, 55, 99)\nRATES = (0.10, 0.20)\nMODELS = ("RF", "XGB", "LGB", "MLP")\n\n\n@dataclass\nclass Dataset:\n    name: str\n    frame: pd.DataFrame\n    features: list[str]\n    source: Path\n    exploratory: bool = False\n\n\ndef load_adult(path: Path) -> Dataset:\n    columns = ["age", "workclass", "fnlwgt", "education", "education-num",\n               "marital-status", "occupation", "relationship", "race", "sex",\n               "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]\n    if path.suffix.lower() == ".zip":\n        with zipfile.ZipFile(path) as zf:\n            members = [n for n in zf.namelist() if n.lower().endswith("adult.data")]\n            if not members:\n                raise ValueError(f"{path} does not contain adult.data")\n            with zf.open(members[0]) as src:\n                frame = pd.read_csv(src, header=None, names=columns,\n                                    na_values=["?", " ?"], skipinitialspace=True)\n    else:\n        frame = pd.read_csv(path, header=None, names=columns,\n                            na_values=["?", " ?"], skipinitialspace=True)\n    frame = frame.dropna().reset_index(drop=True)\n    frame["label"] = (frame["income"].astype(str).str.strip().str.replace(".", "", regex=False) == ">50K").astype(int)\n    frame["sensitive"] = (frame["sex"].astype(str).str.strip() == "Male").astype(int)\n    frame = frame.drop(columns=["income", "sex"])\n    feats = [c for c in frame.columns if c not in ("label", "sensitive")]\n    return Dataset("Adult Income", frame, feats, path)\n\n\ndef load_compas(path: Path) -> Dataset:\n    frame = pd.read_csv(path)\n    frame = frame[(frame["days_b_screening_arrest"] >= -30) &\n                  (frame["days_b_screening_arrest"] <= 30) &\n                  (frame["is_recid"] != -1) & (frame["c_charge_degree"] != "O") &\n                  (frame["score_text"] != "N/A")].copy()\n    keep = ["age", "c_charge_degree", "race", "sex", "priors_count",\n            "days_b_screening_arrest", "juv_fel_count", "juv_misd_count",\n            "juv_other_count", "c_days_from_compas", "is_recid"]\n    frame = frame[keep].dropna().copy()\n    frame["label"] = frame["is_recid"].astype(int)\n    frame["sensitive"] = (~(frame["race"] == "African-American")).astype(int)\n    frame = frame[frame["race"].isin(["African-American", "Caucasian"])].copy().reset_index(drop=True)\n    feats = [c for c in frame.columns if c not in ("label", "sensitive", "is_recid", "race")]\n    return Dataset("COMPAS", frame, feats, path)\n\n\ndef load_taiwan(path: Path) -> Dataset:\n    frame = pd.read_csv(path)\n    frame.columns = frame.columns.str.strip().str.lower().str.replace(" ", "_")\n    if "id" in frame:\n        frame = frame.drop(columns="id")\n    targets = [c for c in frame if "default" in c or "payment" in c]\n    if not targets:\n        raise ValueError("Taiwan CSV has no default/payment target column")\n    target = targets[-1]\n    frame["label"] = frame[target].astype(int)\n    if target != "label":\n        frame = frame.drop(columns=target)\n    sex = next((c for c in frame if "sex" in c), None)\n    if sex is None:\n        raise ValueError("Taiwan CSV has no sex column")\n    frame["sensitive"] = (frame[sex] == 1).astype(int)\n    frame = frame.drop(columns=sex).dropna().reset_index(drop=True)\n    feats = [c for c in frame if c not in ("label", "sensitive")]\n    return Dataset("Taiwan Credit", frame, feats, path)\n\n\ndef load_german_openml() -> Dataset:\n    """Load the notebook\'s OpenML data_id=31 (credit-g), marked exploratory."""\n    from sklearn.datasets import fetch_openml\n    frame = fetch_openml(data_id=31, as_frame=True, parser="auto").frame.copy()\n    frame.columns = frame.columns.str.lower().str.replace("-", "_")\n    if "personal_status" not in frame:\n        raise ValueError("OpenML credit-g data_id=31 lacks personal_status")\n    status = frame["personal_status"]\n    female = status.isin(["A92", "A95", "female div/dep/mar",\n                          "female : divorced/separated/married", "female single"])\n    frame["sensitive"] = (~female).astype(int)\n    frame["label"] = (frame["class"] == "good").astype(int)\n    frame = frame.drop(columns=["class", "personal_status"]).dropna().reset_index(drop=True)\n    feats = [c for c in frame if c not in ("label", "sensitive")]\n    return Dataset("German Credit", frame, feats, Path("OpenML:data_id=31"), True)\n\n\ndef preprocess(data: Dataset, seed: int):\n    x = data.frame[data.features].copy()\n    y = data.frame["label"].to_numpy()\n    x_train, x_test, y_train, y_test = train_test_split(\n        x, y, test_size=0.30, random_state=seed, stratify=y\n    )\n    categorical = x_train.select_dtypes(include=["object", "category"]).columns\n    for col in categorical:\n        train_values, test_values = x_train[col].astype(str), x_test[col].astype(str)\n        enc = LabelEncoder().fit(train_values)\n        mapping = {value: code for code, value in enumerate(enc.classes_)}\n        x_train[col] = train_values.map(mapping).astype(float)\n        x_test[col] = test_values.map(mapping).fillna(-1).astype(float)\n    scaler = MinMaxScaler()\n    x_train = pd.DataFrame(scaler.fit_transform(x_train), columns=data.features, index=x_train.index)\n    x_test = pd.DataFrame(scaler.transform(x_test), columns=data.features, index=x_test.index)\n    group_test = data.frame.loc[x_test.index, "sensitive"].to_numpy()\n    return x_train, x_test, np.asarray(y_train), np.asarray(y_test), group_test\n\n\ndef build_model(name: str, seed: int, balanced: bool = False):\n    if name == "RF":\n        return RandomForestClassifier(n_estimators=100, random_state=seed,\n                                      n_jobs=-1, class_weight="balanced" if balanced else None)\n    if name == "XGB":\n        options: dict[str, Any] = {}\n        if balanced:\n            options["scale_pos_weight"] = 1.0  # caller replaces with train prevalence ratio\n        return XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=seed,\n                             verbosity=0, use_label_encoder=False, **options)\n    if name == "LGB":\n        return LGBMClassifier(n_estimators=100, random_state=seed, verbosity=-1,\n                              is_unbalance=balanced)\n    if name == "MLP":\n        return MLPClassifier(hidden_layer_sizes=(50, 30), max_iter=500,\n                             early_stopping=True, random_state=seed)\n    if name == "LR":\n        return LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", random_state=seed,\n                                  class_weight="balanced" if balanced else None)\n    raise ValueError(f"Unknown model {name}")\n\n\ndef fit_model(name: str, seed: int, x_train, y_train, balanced: bool = False):\n    model = build_model(name, seed, balanced)\n    if name == "XGB" and balanced:\n        positives = max(1, int(np.sum(y_train == 1)))\n        negatives = int(np.sum(y_train == 0))\n        model.set_params(scale_pos_weight=negatives / positives)\n    model.fit(x_train, y_train)\n    return model\n\n\ndef mcar_mask(x_test: pd.DataFrame, rate: float, seed: int) -> pd.DataFrame:\n    rng = np.random.RandomState(seed)\n    mask = rng.rand(*x_test.shape) < rate\n    masked = x_test.copy()\n    masked[mask] = np.nan\n    return masked\n\n\ndef group_mar_mask(x_test: pd.DataFrame, rate: float, group: np.ndarray, seed: int):\n    group = np.asarray(group)\n    if set(np.unique(group)) != {0, 1}:\n        raise ValueError("Both g0 and g1 must be present for group-conditional MAR")\n    q0 = float(np.mean(group == 0))\n    q1 = 1.0 - q0\n    delta = 0.5 * rate\n    p0 = rate + delta * q1\n    p1 = rate - delta * q0\n    row_p = np.where(group == 0, p0, p1)[:, None]\n    rng = np.random.RandomState(int(seed))\n    mask = rng.rand(*x_test.shape) < row_p\n    masked = x_test.copy()\n    masked[mask] = np.nan\n    return masked\n\n\ndef impute_train_median(x_train, x_test):\n    imp = SimpleImputer(strategy="median")\n    imp.fit(x_train)\n    return imp.transform(x_test)\n\n', encoding='utf-8')
Path('/content/work/external/remasker/model_mae.py').write_bytes(Path('/content/remasker/model_mae.py').read_bytes())
Path('/content/work/external/remasker/utils.py').write_bytes(Path('/content/remasker/utils.py').read_bytes())
Path('/content/work/run_remasker_all_models.py').write_text('"""ReMasker comparison across the four primary classifier families."""\nfrom pathlib import Path\nimport json, platform, sys, time, os\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader, TensorDataset\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import accuracy_score, recall_score\n\nROOT = Path(\'/content\')\nREPO = ROOT / "work" / "external" / "remasker"\nFMG_REPO = ROOT / "work" / "FMG-research"\nsys.path.insert(0, str(FMG_REPO / "analysis"))\nsys.path.insert(0, str(REPO))\nif not hasattr(np, "float"):\n    np.float = np.float64\n\nfrom _primary_core import SEEDS, load_adult, load_compas, load_taiwan, preprocess, build_model, mcar_mask\nimport model_mae\n\ntorch.set_num_threads(4)\nRATE, EPOCHS = 0.20, int(os.getenv(\'EPOCHS\', \'600\'))\nMODELS = ("RF", "XGB", "LGB", "MLP")\nDATASETS = [\n    ("Adult Income", load_adult, Path(\'/content/data/adult.zip\')),\n    ("COMPAS", load_compas, Path(\'/content/data/compas-scores-two-years.csv\')),\n    ("Taiwan Credit", load_taiwan, Path(\'/content/data/taiwan_credit.csv\')),\n]\nOUT_DIR = Path(os.environ.get(\'FMG_OUT_DIR\', str(ROOT / \'outputs\')))\nOUT_DIR.mkdir(parents=True, exist_ok=True)\nOUT_CSV = OUT_DIR / "remasker_all_models_summary.csv"\nOUT_JSON = OUT_DIR / "remasker_all_models_results.json"\n\n\ndef metrics(model, x, y, group):\n    pred = model.predict(x)\n    return {"accuracy": float(accuracy_score(y, pred)),\n            "recall_g0": float(recall_score(y[group == 0], pred[group == 0], zero_division=0)),\n            "recall_g1": float(recall_score(y[group == 1], pred[group == 1], zero_division=0))}\n\n\ndef derived(m):\n    m = dict(m); m["focus_recall"] = (m["recall_g0"] + m["recall_g1"]) / 2.0; return m\n\n\ndef fit_remasker(x_train, x_masked, seed):\n    torch.manual_seed(seed); np.random.seed(seed)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    train = torch.tensor(x_train.to_numpy(), dtype=torch.float32)\n    train_mask = torch.ones_like(train)\n    observed = ~x_masked.isna().to_numpy()\n    masked = torch.tensor(x_masked.fillna(0.0).to_numpy(), dtype=torch.float32)\n    mask = torch.tensor(observed.astype(np.float32), dtype=torch.float32)\n    model = model_mae.MaskedAutoencoder(rec_len=train.shape[1], embed_dim=32, depth=4, num_heads=4,\n        decoder_embed_dim=32, decoder_depth=2, decoder_num_heads=4, mlp_ratio=4.0,\n        norm_layer=torch.nn.LayerNorm, encode_func="linear").to(device)\n    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95))\n    loader = DataLoader(TensorDataset(train, train_mask), batch_size=256, shuffle=True)\n    t0 = time.perf_counter(); model.train()\n    for _ in range(EPOCHS):\n        for xb, mb in loader:\n            xb, mb = xb.to(device), mb.to(device)\n            opt.zero_grad(set_to_none=True)\n            loss, _, _, _ = model(xb.unsqueeze(1), mb, mask_ratio=0.5)\n            if not torch.isfinite(loss): raise RuntimeError(f"Non-finite loss at seed {seed}")\n            loss.backward(); opt.step()\n    fit_s = time.perf_counter() - t0\n    model.eval(); chunks = []\n    with torch.no_grad():\n        for i in range(0, len(masked), 512):\n            xb, mb = masked[i:i+512].to(device), mask[i:i+512].to(device)\n            _, pred, _, _ = model(xb.unsqueeze(1), mb, mask_ratio=0.0)\n            chunks.append((mb * xb + (1.0 - mb) * pred.squeeze(2)).cpu().numpy())\n    return pd.DataFrame(np.vstack(chunks), columns=x_train.columns), fit_s\n\n\ndef run_one(data, seed):\n    xtr, xte, ytr, yte, group = preprocess(data, seed)\n    imp = SimpleImputer(strategy="median").fit(xtr)\n    xtr = pd.DataFrame(imp.transform(xtr), columns=xtr.columns)\n    xte_clean = pd.DataFrame(imp.transform(xte), columns=xte.columns)\n    masked_raw = mcar_mask(xte, RATE, seed + 10000)\n    xte_median = pd.DataFrame(imp.transform(masked_raw), columns=xte.columns)\n    xte_remasker, remasker_fit_s = fit_remasker(xtr, masked_raw, seed)\n    rows = []\n    for name in MODELS:\n        t0 = time.perf_counter(); clf = build_model(name, seed); clf.fit(xtr, ytr); classifier_fit_s = time.perf_counter() - t0\n        clean, median, remask = map(lambda x: derived(metrics(clf, x, yte, group)), (xte_clean, xte_median, xte_remasker))\n        rows.append({"dataset": data.name, "model": name, "seed": seed, "mask_rate": RATE,\n            "clean": clean, "median_imputation": median, "remasker": remask,\n            "median_fmg": clean["focus_recall"] - median["focus_recall"],\n            "remasker_fmg": clean["focus_recall"] - remask["focus_recall"],\n            "median_accuracy_loss": clean["accuracy"] - median["accuracy"],\n            "remasker_accuracy_loss": clean["accuracy"] - remask["accuracy"],\n            "classifier_fit_s": classifier_fit_s, "remasker_fit_s": remasker_fit_s,\n            "train_rows": len(xtr), "test_rows": len(xte), "features": xte.shape[1]})\n    return rows\n\n\ndef main():\n    taiwan_csv = Path(\'/content/data/taiwan_credit.csv\')\n    if not taiwan_csv.exists():\n        taiwan_csv.parent.mkdir(parents=True, exist_ok=True)\n        pd.read_excel(Path(\'/content/data/default of credit card clients.xls\'), header=1).to_csv(taiwan_csv, index=False)\n    datasets = [loader(path) for _, loader, path in DATASETS]\n    rows = []\n    if OUT_JSON.exists():\n        try:\n            rows = json.loads(OUT_JSON.read_text(encoding=\'utf-8\')).get(\'rows\', [])\n            print(f\'Resuming from {len(rows)} saved model rows\', flush=True)\n        except Exception as exc:\n            print(f\'Could not read prior checkpoint: {exc}\', flush=True)\n    completed = {(r.get(\'dataset\'), int(r.get(\'seed\'))) for r in rows if r.get(\'dataset\') is not None and r.get(\'seed\') is not None}\n    for data in datasets:\n        for seed in SEEDS:\n            if (data.name, int(seed)) in completed:\n                print(f\'skipping completed {data.name} seed={seed}\', flush=True)\n                continue\n            print(f"running ReMasker {data.name} seed={seed}", flush=True)\n            rows.extend(run_one(data, seed))\n            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)\n            OUT_JSON.write_text(json.dumps({\'protocol\': {\'external\': \'official ReMasker\', \'models\': list(MODELS), \'seeds\': list(SEEDS), \'mask_rate\': RATE, \'epochs\': EPOCHS, \'test_only_masking\': True}, \'environment\': {\'python\': sys.version, \'platform\': platform.platform(), \'torch\': torch.__version__, \'cuda\': torch.cuda.is_available()}, \'rows\': rows}, indent=2), encoding=\'utf-8\')\n            print(f\'checkpoint saved after {data.name} seed={seed}\', flush=True)\n    summary = pd.DataFrame(rows)\n    OUT_JSON.write_text(json.dumps({"protocol": {"external": "official ReMasker", "models": list(MODELS), "seeds": list(SEEDS), "mask_rate": RATE, "epochs": EPOCHS, "test_only_masking": True}, "environment": {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__}, "rows": rows}, indent=2), encoding="utf-8")\n    print(summary.groupby(["dataset", "model"], as_index=False).agg(median_fmg_mean=("median_fmg", "mean"), remasker_fmg_mean=("remasker_fmg", "mean"), remasker_fmg_sd=("remasker_fmg", "std"), remasker_fit_s_mean=("remasker_fit_s", "mean")).to_string(index=False))\n\n\nif __name__ == "__main__": main()\n', encoding='utf-8')
print('runner prepared; CUDA available:', __import__('torch').cuda.is_available())


runner prepared; CUDA available: True


In [4]:
# Run the stronger external comparison. A partial CSV is saved after every seed.
!python /content/work/run_remasker_all_models.py


Resuming from 120 saved model rows
skipping completed Adult Income seed=42
skipping completed Adult Income seed=123
skipping completed Adult Income seed=456
skipping completed Adult Income seed=1
skipping completed Adult Income seed=7
skipping completed Adult Income seed=13
skipping completed Adult Income seed=21
skipping completed Adult Income seed=37
skipping completed Adult Income seed=55
skipping completed Adult Income seed=99
skipping completed COMPAS seed=42
skipping completed COMPAS seed=123
skipping completed COMPAS seed=456
skipping completed COMPAS seed=1
skipping completed COMPAS seed=7
skipping completed COMPAS seed=13
skipping completed COMPAS seed=21
skipping completed COMPAS seed=37
skipping completed COMPAS seed=55
skipping completed COMPAS seed=99
skipping completed Taiwan Credit seed=42
skipping completed Taiwan Credit seed=123
skipping completed Taiwan Credit seed=456
skipping completed Taiwan Credit seed=1
skipping completed Taiwan Credit seed=7
skipping completed T

In [5]:
# Paired exact sign-flip tests, Holm correction, and bootstrap 95% CIs.
from itertools import product
import numpy as np, pandas as pd
from pathlib import Path

out = Path(os.environ.get('FMG_OUT_DIR', '/content/outputs'))
results = pd.read_csv(out/'remasker_all_models_summary.csv')
def sign_flip_p(values):
    values = np.asarray(values, float); obs = abs(values.mean()); n = len(values); hits = 0
    for signs in product([-1, 1], repeat=n):
        if abs(np.mean(values*np.asarray(signs))) >= obs - 1e-15: hits += 1
    return hits / (2**n)
def bootstrap_ci(values, n_boot=10000, seed=2026):
    rng = np.random.RandomState(seed); values = np.asarray(values, float)
    means = np.array([rng.choice(values, len(values), replace=True).mean() for _ in range(n_boot)])
    return float(np.quantile(means,.025)), float(np.quantile(means,.975))
rows = []
for (dataset, model), g in results.groupby(['dataset','model']):
    v = g['remasker_fmg'].to_numpy(); lo, hi = bootstrap_ci(v)
    rows.append({'dataset':dataset,'model':model,'n':len(v),'mean_remasker_fmg':v.mean(),'sd_remasker_fmg':v.std(ddof=1),'ci95_low':lo,'ci95_high':hi,'exact_sign_flip_p':sign_flip_p(v)})
stats = pd.DataFrame(rows).sort_values(['dataset','model']).reset_index(drop=True)
order = np.argsort(stats['exact_sign_flip_p'].to_numpy()); m = len(stats); adj = np.empty(m)
for rank, idx in enumerate(order): adj[idx] = min(1.0, stats.loc[idx,'exact_sign_flip_p']*(m-rank))
stats['holm_adj_p'] = adj
stats.to_csv(out/'remasker_all_models_paired_statistics_colab.csv', index=False)
print(stats.to_string(index=False))


      dataset model  n  mean_remasker_fmg  sd_remasker_fmg  ci95_low  ci95_high  exact_sign_flip_p  holm_adj_p
 Adult Income   LGB 10           0.194946         0.015431  0.185844   0.203729           0.001953    0.023438
 Adult Income   MLP 10           0.025557         0.013924  0.017272   0.033691           0.001953    0.021484
 Adult Income    RF 10           0.105167         0.009857  0.099483   0.110843           0.001953    0.019531
 Adult Income   XGB 10           0.183458         0.020365  0.172291   0.195945           0.001953    0.017578
       COMPAS   LGB 10          -0.022100         0.042541 -0.047744   0.001854           0.142578    0.427734
       COMPAS   MLP 10          -0.000516         0.021007 -0.011866   0.012452           0.945312    0.945312
       COMPAS    RF 10           0.003652         0.028319 -0.013499   0.019777           0.679688    1.000000
       COMPAS   XGB 10           0.026935         0.049881 -0.001974   0.055622           0.123047    0.492188
T

In [6]:
# Confirm persistent Drive outputs.
from pathlib import Path
dest = Path(os.environ.get('FMG_OUT_DIR', '/content/outputs'))
print('Saved:', [p.name for p in dest.glob('remasker_all_models*')])


Saved: ['remasker_all_models_summary.csv', 'remasker_all_models_results.json', 'remasker_all_models_paired_statistics_colab.csv']


## Interpretation guardrails

Only use the Colab results as a manuscript update if the environment, seeds, mask rate, test-only masking, and model definitions match the protocol. Report mixed or null findings honestly. DiffPuter remains a separate optional comparator and should not be implied by this ReMasker run.


## Optional Q2 strengthening run: all-model type-aware preprocessing

This is a separate, resumable sensitivity analysis for the remaining preprocessing concern. It evaluates 3 datasets × 4 classifier families × 10 fixed seeds × 3 raw-feature imputation pipelines at 20% MCAR. It does not overwrite the ReMasker outputs. Run it only after the data-preparation/helper cells have completed.

The result is descriptive and must not be presented as a universal preprocessing ranking. A partial CSV is saved after every model-seed-pipeline result.


In [7]:
# Write and run the resumable all-model type-aware sensitivity script.
# It assumes the preceding setup cells created /content/data and /content/work/FMG-research/analysis/_primary_core.py.
from pathlib import Path
script = '"""Resumable all-model type-aware FMG sensitivity.\n\nRuns 3 datasets x 4 models x 10 fixed seeds x 3 raw-feature imputation\npipelines at 20% MCAR. Masking is applied to raw test cells only; all\nimputers, encoders, and scalers are fitted on clean training data only.\n"""\nfrom __future__ import annotations\nimport json, os, platform, sys, time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.impute import SimpleImputer, KNNImputer\nfrom sklearn.experimental import enable_iterative_imputer  # noqa: F401\nfrom sklearn.impute import IterativeImputer\nfrom sklearn.preprocessing import LabelEncoder, MinMaxScaler\nfrom sklearn.metrics import accuracy_score, recall_score\n\nROOT=Path(\'/content\')\nREPO=ROOT/\'work\'/\'FMG-research\'\nsys.path.insert(0,str(REPO/\'analysis\'))\nfrom _primary_core import SEEDS, load_adult, load_compas, load_taiwan, build_model\n\nRATE=0.20\nMODELS=(\'RF\',\'XGB\',\'LGB\',\'MLP\')\nSTRATEGIES=(\'median_mode\',\'knn_mode\',\'iterative_mode\')\nDATASETS=((\'Adult Income\',load_adult,ROOT/\'data\'/\'adult.zip\'),(\'COMPAS\',load_compas,ROOT/\'data\'/\'compas-scores-two-years.csv\'),(\'Taiwan Credit\',load_taiwan,ROOT/\'data\'/\'taiwan_credit.csv\'))\nOUT=Path(os.environ.get(\'FMG_TYPEAWARE_OUT\',str(ROOT/\'outputs\'/\'typeaware_all_models\')))\nOUT.mkdir(parents=True,exist_ok=True)\nCSV=OUT/\'typeaware_all_models_per_seed.csv\'\nJSON=OUT/\'typeaware_all_models_manifest.json\'\n\ndef mask_raw(x,rate,seed):\n    rng=np.random.RandomState(int(seed)+10000)\n    a=x.copy()\n    m=rng.rand(*a.shape)<rate\n    a=a.astype(object)\n    a[m]=np.nan\n    return a\n\ndef transform_fit(xtr,xte_clean,xte_masked,strategy):\n    xtr=xtr.copy(); clean=xte_clean.copy(); masked=xte_masked.copy()\n    cats=list(xtr.select_dtypes(include=[\'object\',\'category\']).columns)\n    nums=[c for c in xtr.columns if c not in cats]\n    if nums:\n        if strategy==\'median_mode\': imp=SimpleImputer(strategy=\'median\')\n        elif strategy==\'knn_mode\': imp=KNNImputer(n_neighbors=5)\n        elif strategy==\'iterative_mode\': imp=IterativeImputer(max_iter=10,random_state=42)\n        else: raise ValueError(strategy)\n        xtr[nums]=imp.fit_transform(xtr[nums])\n        clean[nums]=imp.transform(clean[nums]); masked[nums]=imp.transform(masked[nums])\n    if cats:\n        cimp=SimpleImputer(strategy=\'most_frequent\')\n        xtr[cats]=cimp.fit_transform(xtr[cats])\n        clean[cats]=cimp.transform(clean[cats]); masked[cats]=cimp.transform(masked[cats])\n        for c in cats:\n            enc=LabelEncoder().fit(xtr[c].astype(str))\n            mapping={v:i for i,v in enumerate(enc.classes_)}\n            xtr[c]=xtr[c].astype(str).map(mapping).astype(float)\n            clean[c]=clean[c].astype(str).map(mapping).fillna(-1).astype(float)\n            masked[c]=masked[c].astype(str).map(mapping).fillna(-1).astype(float)\n    scaler=MinMaxScaler().fit(xtr)\n    return scaler.transform(xtr),scaler.transform(clean),scaler.transform(masked)\n\ndef metrics(model,x,y,g):\n    pred=model.predict(x); f=g==0\n    return accuracy_score(y,pred),recall_score(y[f],pred[f],zero_division=0)\n\ndef main():\n    rows=[]\n    if CSV.exists(): rows=pd.read_csv(CSV).to_dict(\'records\'); print(\'resuming\',len(rows),\'rows\',flush=True)\n    done={(r[\'dataset\'],r[\'model\'],int(r[\'seed\']),r[\'strategy\']) for r in rows}\n    for dsname,loader,path in DATASETS:\n        data=loader(path)\n        raw=data.frame[data.features].copy()\n        y=data.frame[\'label\'].to_numpy(); groups=data.frame[\'sensitive\'].to_numpy()\n        for seed in SEEDS:\n            tr,te=train_test_split(np.arange(len(raw)),test_size=.30,random_state=int(seed),stratify=y)\n            xtr=raw.iloc[tr].copy(); xte=raw.iloc[te].copy(); ytr=y[tr]; yte=y[te]; gte=groups[te]\n            masked=mask_raw(xte,RATE,int(seed))\n            for strategy in STRATEGIES:\n                Xtr,Xclean,Xmask=transform_fit(xtr,xte,xte.copy() if False else masked,strategy)\n                for model_name in MODELS:\n                    key=(dsname,model_name,int(seed),strategy)\n                    if key in done: continue\n                    t0=time.perf_counter(); model=build_model(model_name,int(seed)); model.fit(Xtr,ytr); fit_s=time.perf_counter()-t0\n                    ac,rc=metrics(model,Xclean,yte,gte); am,rm=metrics(model,Xmask,yte,gte)\n                    rows.append({\'dataset\':dsname,\'model\':model_name,\'seed\':int(seed),\'strategy\':strategy,\'rate\':RATE,\'accuracy_clean\':ac,\'accuracy_masked\':am,\'recall_g0_clean\':rc,\'recall_g0_masked\':rm,\'accuracy_loss\':ac-am,\'recall_g0_loss\':rc-rm,\'fmg\':(rc-rm)-(ac-am),\'fit_seconds\':fit_s,\'train_rows\':len(tr),\'test_rows\':len(te),\'features\':len(data.features)})\n                    pd.DataFrame(rows).to_csv(CSV,index=False)\n                    print(dsname,seed,strategy,model_name,\'FMG\',round(rows[-1][\'fmg\'],5),flush=True)\n    manifest={\'protocol\':{\'rate\':RATE,\'seeds\':list(SEEDS),\'models\':list(MODELS),\'strategies\':list(STRATEGIES),\'test_only_raw_cell_mask\':True,\'training_fitted_imputation\':True,\'rows_expected\':3*4*10*3},\'environment\':{\'python\':sys.version,\'platform\':platform.platform(),\'numpy\':np.__version__},\'output\':str(CSV)}\n    JSON.write_text(json.dumps(manifest,indent=2),encoding=\'utf-8\')\n    out=pd.DataFrame(rows); print(out.groupby([\'dataset\',\'model\',\'strategy\'],as_index=False).agg(mean_fmg=(\'fmg\',\'mean\'),sd_fmg=(\'fmg\',\'std\'),positive_splits=(\'fmg\',lambda x:int((x>0).sum())),n=(\'fmg\',\'size\')).to_string(index=False))\nif __name__==\'__main__\': main()\n'
Path('/content/work/run_typeaware_all_models.py').write_text(script, encoding='utf-8')
!python /content/work/run_typeaware_all_models.py


Adult Income 42 median_mode RF FMG 0.00639
Adult Income 42 median_mode XGB FMG 0.0168
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Adult Income 42 median_mode LGB FMG 0.00428
Adult Income 42 median_mode MLP FMG 0.0476
Adult Income 42 knn_mode RF FMG 0.0197
Adult Income 42 knn_mode XGB FMG 0.05317
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  wa